# loss-item-scalar-extract — faded example 3: Build a Step-Loss Buffer Using .item() vs .detach().clone()

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `loss-item-scalar-extract`. The last cell reports your progress on the `PyTorch: loss.item() scalar extract` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: loss.item() scalar extract` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`loss-item-scalar-extract`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "loss-item-scalar-extract"
DD_SUBTOPIC = "PyTorch: loss.item() scalar extract"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In training loops you often want both a running scalar for live display AND a tensor buffer for later analysis. `.item()` gives you the former: a plain Python float for f-strings and wandb. `.detach().clone()` gives you the latter: a tensor detached from the graph, safe to stack later without leaking the autograd graph. Both patterns are needed in a complete training scaffold.

## Faded exercise 3

Complete the function below. The loop builds both buffers. Fill in the single blanked line that extracts the float for the `scalar_log`.

Do not change any other part of the function.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def build_dual_log(n_steps: int):
    """
    Run n_steps of a fake training loop and return two logs:
    - scalar_log: list of Python floats (for wandb/print)
    - tensor_log: stacked (n_steps,) tensor (for later analysis)
    """
    scalar_log = []
    tensor_buf = []
    for i in range(n_steps):
        t.manual_seed(i + 300)
        x = t.randn(4, requires_grad=True)
        loss = (x ** 2).mean()

        step_float = loss.item()
        scalar_log.append(step_float)
        tensor_buf.append(loss.detach().clone())

    return {
        'scalar_log': scalar_log,
        'tensor_log': t.stack(tensor_buf),
    }

result = build_dual_log(4)
print("scalar_log:", result['scalar_log'])
print("tensor_log:", result['tensor_log'])
print("types:", [type(v).__name__ for v in result['scalar_log']])


import torch as t

def _test():
    result = build_dual_log(4)
    scalar_log = result['scalar_log']
    tensor_log = result['tensor_log']

    assert len(scalar_log) == 4
    assert all(isinstance(v, float) for v in scalar_log), "All scalar_log entries must be Python floats"

    assert tensor_log.shape == (4,), f"tensor_log shape should be (4,), got {tensor_log.shape}"

    # Scalar and tensor logs must agree numerically
    for i, (sf, tv) in enumerate(zip(scalar_log, tensor_log)):
        assert abs(sf - tv.item()) < 1e-6, f"Step {i}: scalar_log and tensor_log disagree"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def build_dual_log(n_steps: int):
    """
    Run n_steps of a fake training loop and return two logs:
    - scalar_log: list of Python floats (for wandb/print)
    - tensor_log: stacked (n_steps,) tensor (for later analysis)
    """
    scalar_log = []
    tensor_buf = []
    for i in range(n_steps):
        t.manual_seed(i + 300)
        x = t.randn(4, requires_grad=True)
        loss = (x ** 2).mean()

        step_float = loss.item()
        scalar_log.append(step_float)
        tensor_buf.append(loss.detach().clone())

    return {
        'scalar_log': scalar_log,
        'tensor_log': t.stack(tensor_buf),
    }

result = build_dual_log(4)
print("scalar_log:", result['scalar_log'])
print("tensor_log:", result['tensor_log'])
print("types:", [type(v).__name__ for v in result['scalar_log']])
```
</details>